In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import os
from transformers import AutoImageProcessor, AutoModelForImageClassification
import torch.nn as nn
import cv2

# Model name
MODEL_NAME = "trpakov/vit-face-expression"

# Load processor and model
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageClassification.from_pretrained(MODEL_NAME)

class_names = ['truth', 'lie']

model.classifier = nn.Linear(model.config.hidden_size, len(class_names))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load trained model
model.load_state_dict(torch.load("best_model_weights.pt", map_location=torch.device('cpu')))
model.eval()
model.to(device)


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermed

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)  # or use processor.image_mean/std
])

In [ ]:
def load_images_from_folder(folder):
    images = []
    for filename in os.listdir(folder):
        if filename.lower().endswith((".jpg", ".png", ".jpeg")):
            img = Image.open(os.path.join(folder, filename)).convert("RGB")
            img_tensor = transform(img)
            images.append(img_tensor)
    return torch.stack(images)

In [ ]:
def predict_sequence(image_batch):
    image_batch = image_batch.to(device)
    with torch.no_grad():
        outputs = model(pixel_values=image_batch).logits
        probs = torch.softmax(outputs, dim=1)
    return probs


In [ ]:
def classify_subject(image_folder):
    images = load_images_from_folder(image_folder)
    probs = predict_sequence(images)

    avg_probs = probs.mean(dim=0)  # average over all photos
    predicted_idx = avg_probs.argmax().item()
    predicted_label = class_names[predicted_idx]

    print(f"🧠 Final Decision: {predicted_label.upper()} (Confidence: {avg_probs[predicted_idx]:.2f})")
    return predicted_label, avg_probs


In [ ]:
def extract_frames_per_second(video_path, output_folder, fps_interval=1):
    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(fps * fps_interval)  # e.g., every 30 frames for 30fps

    frame_count = 0
    saved_count = 0

    while True:
        success, frame = cap.read()
        if not success:
            break

        if frame_count % frame_interval == 0:
            frame_filename = os.path.join(output_folder, f"frame_{saved_count:04d}.jpg")
            cv2.imwrite(frame_filename, frame)
            saved_count += 1

        frame_count += 1

    cap.release()
    print(f"✅ Extracted {saved_count} frames to: {output_folder}")


In [ ]:
extract_frames_per_second("/content/IMG_4745.MOV", 'ExtractedFrames', fps_interval=1)

✅ Extracted 45 frames to: ExtractedFrames


In [ ]:
folder_path = "/content/ExtractedFrames"
classify_subject(folder_path)

🧠 Final Decision: LIE (Confidence: 0.57)


('lie', tensor([0.4318, 0.5682]))